## 01 - EDF to spectrogram

Converts one channel of an `.edf` recording to a raw CSV (reused when cached) and plots its time-frequency spectrogram with event lines.

Writes `cache/<animal>/csvs/<label>_raw_ch<c>.csv` and, under `outputs/01_edf_to_spectrogram/`, `<label>_ch<c>_spectrogram.png/.svg` and `run_info.json`.

### Variables

In [ ]:
EDF_FOLDER = "../../Completed_Analyses/Emergence/raw_edfs"  # folder holding the .edf files (relative to this notebook)
EDF = "EmergRev_3Perc_Induction_PEEG02M_2026-08-27_15_49_57_export.edf"  # .edf file name inside EDF_FOLDER (a unique part of the name also works)
ANIMAL_ID = "PEEG02M"  # animal id: first part of the label and name of the cache sub-folder
CONDITION = "Induction"  # condition name: second part of the label and name of the raw CSV column
RECORDING_NUMBER = 1  # session number: label = <ANIMAL_ID>_<CONDITION>_R<RECORDING_NUMBER>
CHANNEL = 0  # EDF channel to extract and plot (0/1 = EEG, 2 = EMG)
EVENTS = {"onset": 1800}  # named events in recording-clock seconds; each is drawn as a dashed vertical line
EVENT_COLORS = {"onset": "white"}  # line colour per event name (events not listed are drawn white)
EXTRA_LINES = {"+15 min": 900}  # lines added after plotting (step 4): seconds on the plotted axis, i.e. relative to ALIGN_EVENT
EXTRA_COLOR = "red"  # colour of the EXTRA_LINES
LEGEND_BBOX = (1.3, 1)  # legend anchor in axes fractions; places it right of the colour bar
ALIGN_EVENT = "onset"  # event placed at x = 0; None keeps recording-clock time
WINDOW = (-300, 3600)  # x-axis range in seconds relative to ALIGN_EVENT; an end of None shows the whole recording
FREQ_RANGE = (0, 20)  # y-axis frequency range (Hz); (0, 30) also shows beta
CLIM = (-90, -110)  # colour limits in dB/Hz as (high, low); raise the low value to brighten weak power
FIGSIZE = (5, 2.5)  # figure size in inches
TICK_STEP = 600  # x tick spacing in seconds
N_FFT = 10000  # spectrogram window in samples (10000 at 400 Hz = 25 s windows, 50% overlap)
NOTCH_FREQS = None  # frequencies (Hz) notch-filtered while extracting, e.g. [60]; None = unfiltered
OVERWRITE = False  # True re-extracts the raw CSV even when it is cached (needed after changing NOTCH_FREQS)
WORK_DIR = "cache"  # processing cache (raw CSV), reused between runs
OUTPUT_DIR = "outputs/01_edf_to_spectrogram"  # figures and run info; files are never overwritten (_1, _2, ... are appended)
FIGURE_FORMATS = ("png", "svg")  # one file per format for each figure
DPI = 300  # resolution of the png

In [ ]:
from pathlib import Path  # path handling
import sys  # module search path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ecog").is_dir())  # repo root = first parent folder containing ecog/
sys.path.insert(0, str(ROOT))  # makes `import ecog` work without installing anything
import os  # path normalising
EDF_DIR = Path(os.path.abspath(EDF_FOLDER))  # absolute EDF folder with any `..` collapsed (relative paths start at this notebook's folder)
import mne  # EDF reader used by ecog
mne.set_log_level("WARNING")  # hides the EDF reader's progress messages
import pandas as pd  # reads the raw CSV to report its length
from ecog import Recording, extractRecording, saveFigure, writeRunInfo  # recording description, EDF -> CSV, no-overwrite saving
from ecog.plots import plotRecordingSpectrogram, annotateTimeline  # spectrogram and event-line helpers
WORK = Path(WORK_DIR)  # cache folder
OUT = Path(OUTPUT_DIR)  # output folder

### 1. Describe the recording

In [ ]:
# everything the processing functions need to know about this recording
rec = Recording(animal_id=ANIMAL_ID, condition=CONDITION, recording_number=RECORDING_NUMBER, edf=EDF, edf_folder=EDF_DIR,
                channel_index=CHANNEL, events=EVENTS, event_colors=EVENT_COLORS, n_fft=N_FFT, notch_freqs=NOTCH_FREQS,
                clim=CLIM, tick_step=TICK_STEP)
rec.validate()  # stops with one message if the file, channel or events are wrong
rec.label  # label used in every file name

### 2. EDF to raw CSV (skipped when the CSV already exists)

In [ ]:
rawPaths = extractRecording(rec, WORK, overwrite=OVERWRITE)  # {channel: raw CSV}; whole recording in volts, one column named CONDITION
nSamples = len(pd.read_csv(rawPaths[CHANNEL]))  # samples in the CSV
print(rawPaths[CHANNEL], f"{nSamples / rec.fs / 60:.1f} min")  # file written and recording length

### 3. Spectrogram (events in EVENTS are drawn automatically)

In [ ]:
# time-frequency spectrogram of the channel, axis re-zeroed on ALIGN_EVENT and cropped to WINDOW
fig = plotRecordingSpectrogram(rec, CHANNEL, WORK, freq_min=FREQ_RANGE[0], freq_max=FREQ_RANGE[1], align_event=ALIGN_EVENT,
                               start_time=WINDOW[0], end_time=WINDOW[1], figsize=FIGSIZE, clim=CLIM, tick_step=TICK_STEP)

### 4. Add event lines (permanent events belong in EVENTS, in recording-clock seconds)

In [ ]:
# extra dashed lines at x positions of the plotted axis
annotateTimeline(fig.axes[0], events=EXTRA_LINES, tick_step=TICK_STEP, colors={name: EXTRA_COLOR for name in EXTRA_LINES})
fig.axes[0].legend(loc="upper left", bbox_to_anchor=LEGEND_BBOX, facecolor="0.6", framealpha=1)  # one legend for all lines, clear of the colour bar; gray box keeps white lines visible
fig  # show the finished figure

### 5. Save

In [ ]:
# png/svg under outputs/, with an index appended when the name exists
paths = saveFigure(fig, OUT / f"{rec.label}_ch{CHANNEL}_spectrogram", dpi=DPI, formats=FIGURE_FORMATS)
info = writeRunInfo(OUT, settings=dict(edf=EDF, channel=CHANNEL, events=EVENTS, align_event=ALIGN_EVENT, window=WINDOW,
                                       freq_range=FREQ_RANGE, clim=CLIM, n_fft=N_FFT, notch_freqs=NOTCH_FREQS))  # versions and settings
print(paths, info)  # files written